# Phase 3 — preliminary leaderboard

Reads `outputs/eval/leaderboard_phase3.csv` (produced by `scripts/benchmark_pipeline.py`) and renders:
1. Per-metric leaderboard bar charts at each ratio (4× / 5× / 10×).
2. The same, sliced by traditional vs hard.
3. The same, per challenge tag.
4. A method gallery on five curated images.
5. Metric-vs-human correlation, reading `data/human_ranks.json`.

**Reminder:** PSNR and SSIM are *misleading* for diffusion-based methods — they reward blur over invented detail. LPIPS and DISTS are the load-bearing metrics. PSNR/SSIM included for completeness but flagged on every chart.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from upscaler import testset

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL_DIR = REPO_ROOT / "outputs" / "eval"
PHASE3_OUT = EVAL_DIR / "phase3_charts"
PHASE3_OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(EVAL_DIR / "leaderboard_phase3.csv")
print(
    f"loaded {len(df)} rows: {df.method.nunique()} methods × {df.image.nunique()} images × {df.lr_size.nunique()} ratios"
)
df.head()

In [ ]:
METHODS = ["bicubic", "lanczos", "realesrgan", "x4_upscaler", "two_stage"]
METRICS = [
    ("lpips", "LPIPS (lower better)", True),
    ("dists", "DISTS (lower better)", True),
    ("psnr", "PSNR (higher better, MISLEADING for diffusion)", False),
    ("ssim", "SSIM (higher better, MISLEADING for diffusion)", False),
]
RATIOS = [(250, "4×"), (200, "5×"), (100, "10×")]
COLORS = {
    "bicubic": "#999999",
    "lanczos": "#4477AA",
    "realesrgan": "#EE6677",
    "x4_upscaler": "#228833",
    "two_stage": "#CCBB44",
}

## (a) Overall leaderboard — full 60-image set

In [ ]:
from IPython.display import Image as IPImage


def bar_chart_per_metric(
    data: pd.DataFrame,
    title: str,
    out_path: Path,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(title, fontsize=14)
    for ax, (metric, label, _lower_better) in zip(axes.flat, METRICS, strict=True):
        agg = data.groupby(["method", "lr_size"])[metric].mean().unstack("lr_size")
        agg = agg.reindex(METHODS)
        x = np.arange(len(METHODS))
        width = 0.27
        for i, (lr, label_r) in enumerate(RATIOS):
            offset = (i - 1) * width
            ax.bar(x + offset, agg[lr].values, width=width, label=label_r)
        ax.set_xticks(x)
        ax.set_xticklabels(METHODS, rotation=20, ha="right")
        ax.set_ylabel(metric.upper())
        ax.set_title(label, fontsize=11)
        ax.legend(loc="upper right", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        if metric == "psnr":
            # Trim the y range so PSNR's tens-of-dB scale doesn't squish info.
            ax.set_ylim(bottom=max(0, agg.values.min() - 1))
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close(fig)


bar_chart_per_metric(df, "Overall (full 60-image set)", PHASE3_OUT / "overall.png")
IPImage(filename=str(PHASE3_OUT / "overall.png"))

## (b) Traditional vs hard

In [ ]:
for cat in ("traditional", "hard"):
    sub = df[df.category == cat]
    bar_chart_per_metric(
        sub, f"{cat.capitalize()} subset (n={sub.image.nunique()})", PHASE3_OUT / f"{cat}.png"
    )

from IPython.display import Image as IPImage  # noqa: E402

IPImage(filename=str(PHASE3_OUT / "hard.png"))

## (c) Per challenge tag

An image with multiple challenge tags shows up in every relevant slice — the slices intentionally overlap.

In [ ]:
ts = testset.load(REPO_ROOT / "data" / "test_images")
tagged: dict[str, set[str]] = {tag: set() for tag in ts.challenges}
for img in ts:
    for tag in img.challenges:
        tagged[tag].add(img.name)

for tag in sorted(tagged):
    sub = df[df.image.isin(tagged[tag])]
    if sub.empty:
        continue
    bar_chart_per_metric(
        sub,
        f"Hard / {tag}  (n={sub.image.nunique()})",
        PHASE3_OUT / f"challenge_{tag}.png",
    )

from IPython.display import Image as IPImage  # noqa: E402

IPImage(filename=str(PHASE3_OUT / "challenge_text.png"))

## Method gallery — 5 representative images at 5× (200→1000)

Each row: HR reference + every method's output. Use this to eyeball whether the LPIPS/DISTS rankings match what you actually see.

In [ ]:
GALLERY_IMAGES = [
    "California_Wildflowers",
    "Paris_Street",
    "Duomo_fine-arch",
    "Forest_hf-texture",
    "NewYork_Street_Text",
]
GALLERY_LR = 200
CROP_SIZE = 400

PHASE1_CACHE = REPO_ROOT / "outputs" / "phase1" / "upscales"
PHASE2_CACHE = REPO_ROOT / "outputs" / "phase2" / "upscales"


def gallery_path(method: str, stem: str, lr: int) -> Path:
    if method == "two_stage":
        return PHASE2_CACHE / f"{stem}_{lr}_d030_s20.jpg"
    return PHASE1_CACHE / f"{stem}_{lr}_{method}.jpg"


def render_gallery() -> Path:
    cols = ["HR", *METHODS]
    fig, axes = plt.subplots(
        len(GALLERY_IMAGES), len(cols), figsize=(2.6 * len(cols), 2.6 * len(GALLERY_IMAGES))
    )
    cx = (1000 - CROP_SIZE) // 2
    crop = (cx, cx, cx + CROP_SIZE, cx + CROP_SIZE)

    for r, stem in enumerate(GALLERY_IMAGES):
        hr = (
            Image.open(REPO_ROOT / "data" / "test_images" / f"{stem}.jpg").convert("RGB").crop(crop)
        )
        axes[r, 0].imshow(hr)
        axes[r, 0].axis("off")
        if r == 0:
            axes[r, 0].set_title("HR", fontsize=10)
        axes[r, 0].text(
            -0.05,
            0.5,
            stem,
            rotation=0,
            ha="right",
            va="center",
            transform=axes[r, 0].transAxes,
            fontsize=9,
        )
        for c, method in enumerate(METHODS, start=1):
            cell = Image.open(gallery_path(method, stem, GALLERY_LR)).convert("RGB").crop(crop)
            axes[r, c].imshow(cell)
            axes[r, c].axis("off")
            if r == 0:
                axes[r, c].set_title(method, fontsize=10)

    fig.suptitle(f"Gallery — 5×, center {CROP_SIZE}×{CROP_SIZE} crop", fontsize=12)
    fig.tight_layout(rect=[0.03, 0, 1, 0.97])
    out = PHASE3_OUT / "gallery_5x.png"
    fig.savefig(out, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return out


gallery_path_out = render_gallery()
from IPython.display import Image as IPImage  # noqa: E402

IPImage(filename=str(gallery_path_out))

## Metric-vs-human correlation

**Brad's task:** rank the 5 methods on each gallery image (best=1, worst=5) and save as `data/human_ranks.json` (template at `data/human_ranks.template.json`). The cell below reads that file and plots Spearman correlation between Brad's ranks and each metric's ranks. If no file is found, the cell prints instructions and exits gracefully.

In [ ]:
from scipy.stats import spearmanr

ranks_path = REPO_ROOT / "data" / "human_ranks.json"
if not ranks_path.is_file():
    print("No data/human_ranks.json found.")
    print(f"  Template: {REPO_ROOT / 'data' / 'human_ranks.template.json'}")
    print("  Fill in best=1..worst=5 ranks for each gallery image, save without .template.")
else:
    payload = json.loads(ranks_path.read_text())
    ranks = payload["ranks"]
    lr = payload.get("lr_size", 200)

    rows = []
    for image, human in ranks.items():
        sub = df[(df.image == image) & (df.lr_size == lr)].set_index("method")
        if sub.empty:
            print(f"warning: no rows for {image} at lr_size={lr}")
            continue
        for metric, _, lower_better in METRICS:
            vals = sub.loc[METHODS, metric]
            metric_rank = vals.rank(ascending=lower_better, method="min").to_dict()
            for m in METHODS:
                rows.append(
                    {
                        "image": image,
                        "method": m,
                        "metric": metric,
                        "human_rank": human[m],
                        "metric_rank": metric_rank[m],
                    }
                )
    corr_df = pd.DataFrame(rows)

    print(f"Spearman correlation (per metric, ranks pooled across {len(ranks)} images):")
    for metric, label, _ in METRICS:
        sub = corr_df[corr_df.metric == metric]
        rho, p = spearmanr(sub.human_rank, sub.metric_rank)
        print(f"  {metric:>5}: rho={rho:+.3f}  p={p:.3g}   ({label})")

    fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 4), sharey=True)
    for ax, (metric, _label, _lower_better) in zip(axes, METRICS, strict=True):
        sub = corr_df[corr_df.metric == metric]
        for m in METHODS:
            mm = sub[sub.method == m]
            ax.scatter(mm.metric_rank, mm.human_rank, label=m, alpha=0.7, s=80)
        rho, _ = spearmanr(sub.human_rank, sub.metric_rank)
        ax.plot([1, 5], [1, 5], "--", color="gray", alpha=0.5)
        ax.set_title(f"{metric.upper()}  rho={rho:+.2f}", fontsize=11)
        ax.set_xlabel("metric rank")
        ax.set_xticks(range(1, 6))
        ax.set_yticks(range(1, 6))
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("human rank (1 = best)")
    axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    fig.suptitle("Metric vs human rank — Spearman rho", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    out_corr = PHASE3_OUT / "human_correlation.png"
    fig.savefig(out_corr, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\nsaved: {out_corr}")

### Result: which metric tracks human perception best?

Spearman ρ between Brad's ranks and each metric's ranks (n=25 method-image pairs over 5 images at 5×):

| metric | ρ | p | interpretation |
|---|---|---|---|
| **LPIPS** | **+0.80** | 1.6e-6 | strongly tracks human ranking — load-bearing metric for the rest of the project |
| DISTS | +0.70 | 9.8e-5 | tracks well; second-best |
| PSNR | **−0.68** | 1.8e-4 | **anti-correlates** — higher PSNR predicts worse human rank |
| SSIM | **−0.54** | 5.3e-3 | also anti-correlates |

**Empirical confirmation that PSNR and SSIM are misleading for diffusion-based upscaling.** They reward conservative pixel-matching (bicubic, Lanczos) and penalize the invented-but-plausible detail that humans actually prefer. We carry LPIPS as the primary metric through Phase 4 and beyond; PSNR/SSIM stay on the leaderboard for completeness only and are flagged on every chart.

## Notes for downstream phases

- Re-run `scripts/benchmark_pipeline.py` after each LoRA training run with `--name leaderboard_phase4`.
- This notebook becomes the dashboard for Phase 4 and Phase 4.6 — only the CSV path needs to change to compare runs.